# C4: fine-tune the Sinhala ASR on OpenSLR SLR52

C4 transcribes the child's answer before measuring it. The service currently uses a public Sinhala fine-tune of XLS-R-300m. This notebook trains the team's own model, as the C4 plan asks, and checks that it is better on held-out speakers:

1. **Data:** downloads SLR52 shards (Sinhala read speech, 478 speakers) and makes **speaker-disjoint, sentence-disjoint** train / dev / test splits.
2. **Baseline:** WER / CER of the service's current model on the test split.
3. **Training:** one of three starting points (`SWARA_C4_BASE`):
   - the current model (continue training, its own vocabulary; the default);
   - `facebook/wav2vec2-xls-r-300m` (a new character head);
   - `facebook/mms-1b-all` (MMS with a new Sinhala adapter; only the adapter and head train).
4. **Evaluation and export:** the same WER / CER on the same test utterances, a report in `training/reports/`, and a model folder the service loads with `C4_CTC_MODEL=<folder>`.

**Settings** (environment variables, with defaults): `SWARA_C4_DATA` (`data/slr52`), `SWARA_C4_SHARDS` (`0`; up to `0123456789abcdef`), `SWARA_C4_BASE`, `SWARA_C4_STEPS` (2000), `SWARA_C4_BATCH` (8), `SWARA_C4_ACCUMULATE` (2), `SWARA_C4_LR` (1e-4; 1e-3 for MMS adapters), `SWARA_C4_MAX_TRAIN` / `SWARA_C4_MAX_EVAL` (0 = all; small numbers for a quick test), `SWARA_C4_DEVICE` (auto), `SWARA_C4_OUT` (`models/c4_asr`), `SWARA_C4_REPORT_DIR` (`training/reports`), `SWARA_C4_NOTE`.

SLR52 is CC BY-SA 4.0: cite Kjartansson et al., SLTU 2018. It is adult read speech, so check the model on the team's child recordings before relying on it for children.

In [ ]:
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # the language service's own code (app/, training/)
from training import asr_finetune as ft, slr52

DATA = Path(os.environ.get("SWARA_C4_DATA", ROOT / "data" / "slr52"))
SHARDS = list(os.environ.get("SWARA_C4_SHARDS", "0"))
CURRENT = "SpideyDLK/wav2vec2-large-xls-r-300m-sinhala-low-LR-part1"  # the service's default (app/config.py)
BASE = os.environ.get("SWARA_C4_BASE", CURRENT)
STEPS = int(os.environ.get("SWARA_C4_STEPS", 2000))
BATCH = int(os.environ.get("SWARA_C4_BATCH", 8))
ACCUMULATE = int(os.environ.get("SWARA_C4_ACCUMULATE", 2))
LR = float(os.environ.get("SWARA_C4_LR", 1e-3 if BASE == ft.MMS else 1e-4))
MAX_TRAIN = int(os.environ.get("SWARA_C4_MAX_TRAIN", 0))
MAX_EVAL = int(os.environ.get("SWARA_C4_MAX_EVAL", 0))
DEVICE = os.environ.get("SWARA_C4_DEVICE") or ft.best_device()
OUT = Path(os.environ.get("SWARA_C4_OUT", ROOT / "models" / "c4_asr"))
REPORT_DIR = Path(os.environ.get("SWARA_C4_REPORT_DIR", ROOT / "training" / "reports"))
NOTE = os.environ.get("SWARA_C4_NOTE", "")
print(f"data {DATA} | shards {''.join(SHARDS)} | base {BASE} | {STEPS} steps x {BATCH}x{ACCUMULATE} | lr {LR} | device {DEVICE}")

## 1. Data: download, index, speaker- and sentence-disjoint splits

In [ ]:
slr52.fetch(DATA, SHARDS)  # resumable; shards already unzipped are skipped
df = slr52.split(slr52.read_index(DATA))
df = df[df["available"]]  # only the shards on disk
print(slr52.summary(df).to_string())
for a, b in [("train", "dev"), ("train", "test"), ("dev", "test")]:
    assert not set(df.loc[df.split == a, "speaker"]) & set(df.loc[df.split == b, "speaker"]), f"speaker in {a} and {b}"
assert not set(df.loc[df.split == "train", "norm"]) & set(df.loc[df.split != "train", "norm"]), "a test sentence is in train"

rows = lambda split, n: df[df.split == split].sample(frac=1, random_state=17).head(n or None).to_dict("records")
train_rows, dev_rows, test_rows = rows("train", MAX_TRAIN), rows("dev", MAX_EVAL), rows("test", MAX_EVAL)
print(len(train_rows), "train /", len(dev_rows), "dev /", len(test_rows), "test utterances used")

## 2. Baseline: the service's current model on the test split

In [ ]:
from transformers import AutoModelForCTC, AutoProcessor

t0 = time.time()
cur_proc, cur_model = AutoProcessor.from_pretrained(CURRENT), AutoModelForCTC.from_pretrained(CURRENT)
baseline = ft.score([r["text"] for r in test_rows], ft.transcribe(cur_model, cur_proc, test_rows, DEVICE))
del cur_model
print("current model on test:", baseline, f"({time.time() - t0:.0f} s)")

## 3. Training

In [ ]:
model, processor, mode = ft.load_model(BASE, OUT, [r["norm"] for r in train_rows])
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"mode {mode}: {trainable:,} trainable parameters")
loss_log = ft.train(model, processor, train_rows, DEVICE, steps=STEPS, batch_size=BATCH, lr=LR, accumulate=ACCUMULATE)

## 4. Evaluation (same test utterances) and export

In [ ]:
import pandas as pd

dev = ft.score([r["text"] for r in dev_rows], ft.transcribe(model, processor, dev_rows, DEVICE))
test = ft.score([r["text"] for r in test_rows], ft.transcribe(model, processor, test_rows, DEVICE))
table = pd.DataFrame({"current service model": baseline, f"fine-tuned ({mode})": test}).T
print("dev:", dev)
table

In [ ]:
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d")
info = {
    "base": BASE, "mode": mode, "shards": SHARDS, "steps": STEPS, "batch": BATCH, "accumulate": ACCUMULATE, "lr": LR,
    "trainUtterances": len(train_rows), "devUtterances": len(dev_rows), "testUtterances": len(test_rows),
    "speakers": {s: int(df.loc[df.split == s, "speaker"].nunique()) for s in ("train", "dev", "test")},
    "baselineTest": baseline, "dev": dev, "test": test, "lossLog": loss_log, "trainedAt": stamp, "note": NOTE,
}
ft.save(model, processor, OUT, info)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / f"c4-asr-{stamp}.json").write_text(json.dumps(info, indent=1, ensure_ascii=False, default=str))
(REPORT_DIR / f"c4-asr-{stamp}.md").write_text(
    f"# C4 ASR fine-tuning ({stamp})\n\nBase `{BASE}` ({mode}), SLR52 shards {''.join(SHARDS)}, {STEPS} steps. {NOTE}\n\n"
    f"Speaker- and sentence-disjoint splits: {info['speakers']} speakers (train / dev / test).\n\n" + table.to_markdown() + "\n")
print("saved", OUT, "and the report in", REPORT_DIR)

In [ ]:
# The service loads the folder with C4_CTC_MODEL=<OUT> (app/asr/ctc.py); check it transcribes a test clip.
from app.asr.ctc import CtcEngine

engine = CtcEngine(str(OUT), "cpu", 20.0)
clip = ft.read_audio(Path(test_rows[0]["path"]))
print("reference:", test_rows[0]["text"])
print("service  :", engine.transcribe(clip, ft.SR)["text"])